# Selection model: how the FantasyGo field picks players (2023-24)

The opponent side of the MVT (`src/selection.py`, read by `src/mvt.py`'s team generator). For player $j$ of a position in
round $t$:

$$\alpha_{j,t} = \exp(\beta_0 + x_{j,t}^T\beta), \qquad c_t \sim \text{DirichletMultinomial}(n_t, \alpha_t)$$

* $c_{j,t}$: the number of FantasyGo entries that picked player $j$ in round $t$, $n_t = \sum_j c_{j,t}$
* $x_{j,t}$: the player's standardized features, known before the deadline (ownership, transfers, last round's
  FantasyGo share, fixture, form)
* $\beta_0$: sets how concentrated the picks are; $\alpha_{j,t}/\sum_k\alpha_{k,t}$ is the expected pick share

One Dirichlet-multinomial per round, $\beta \sim N(0, 0.5)$, $b_0 \sim N(0, 1)$ (the tight priors chosen in selection_checks.ipynb), NUTS. Features are ranked by correlation with the pick share
and de-duplicated, fitted, pruned to those whose 94% HDI excludes 0, and refitted. One fit per gameweek (6-38) on the
rounds before it only, so every check below is out of sample.

Ported from `23-24/optimizer/MVT/beta estimators.ipynb` on `data/23-24/players.csv` (same rows, order and features;
the inputs differ only by floating-point rounding, about $10^{-15}$, so the posteriors agree with that notebook's to
three decimals but not draw for draw).

FantasyGo gameweeks 22-25 hold other rounds' contests (copies of GW15, 16 and 20, and a GW18 single-match contest;
see the README) and are treated as missing: left out of the fits, $\kappa$ and every check below.

Fit with `cd src && python selection.py` (about an hour; cached in `outputs/23-24/selection/`, resumable). This
notebook reads the cache.

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
import selection  # noqa: E402

data, picks = selection.load_data("23-24")
models = selection.fit_all(data)  # from the cache; fits and caches anything missing
est, powers = selection.write_outputs(data, models)  # league_select_probs.csv, league_alpha_draws.pkl
print(data.groupby("position").agg(rows=("fpl_id", "size"), picks=("fg_picks", "sum")).loc[selection.POSITIONS])

## Coefficients of the last fit (standardized features)

In [ ]:
pd.concat({pos: fits[max(fits)]["summary_final"][["mean", "hdi_3%", "hdi_97%"]] for pos, fits in models.items()})

## Sharpening power $\kappa$

$\alpha_j^\kappa/\sum_k\alpha_k^\kappa$, fitted per position and gameweek by the log loss of the actual picks in the
earlier rounds; the team generator uses it. It stays close to 1: the expected shares are already about right (see
the last section for why the simulated field still came out too flat).

In [ ]:
pd.Series(powers).unstack(0)[selection.POSITIONS].round(2).T

## Evaluation

Four questions, each with its own check:

1. **Did the sampler converge?** $\hat R$, bulk ESS and divergences of every fit
2. **Does it predict who is picked, every gameweek?** log loss of the actual picks, round by round, against three
   baselines: every candidate equally likely, FPL ownership and last round's FantasyGo share
3. **Are its probabilities calibrated?** where each player's actual pick count falls in its posterior predictive
   distribution
4. **Does it reproduce how concentrated the picks are?** posterior predictive checks of statistics of the whole
   round, which the expected shares alone cannot test

### 1. Convergence

$\hat R \le 1.01$ and bulk ESS $\ge 400$ are the usual thresholds; any divergence makes a fit suspect.

In [ ]:
selection.convergence(models)

### 2. Who is picked, round by round

* `log_loss`: mean negative log-probability of each actual pick (lower is better)
* `spearman`: rank correlation of the predicted and actual shares
* `top10`: how many of the 10 most-picked players are in the predicted top 10

Then, per baseline: `model_better`, the rounds where the model's log loss is lower; `mean_gain`, the baseline's log
loss minus the model's; `sign_test_p`, a two-sided sign test of `model_better` against half the rounds. The fits of
neighbouring rounds overlap, so the p-values are indicative.

In [ ]:
check = selection.out_of_sample(data, models)
display(check.groupby(["position", "predictor"])[["log_loss", "spearman", "top10"]].mean().round(3)
        .unstack("position")[[(m, p) for m in ["log_loss", "spearman", "top10"] for p in selection.POSITIONS]])
selection.paired_tests(check).round(3)

In [ ]:
ll = check.pivot_table(index=["position", "round"], columns="predictor", values="log_loss")
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharex=True)
for ax, pos in zip(axes, selection.POSITIONS):
    for name in ["model", "fpl_ownership", "last_round_fg"]:
        ax.plot(ll.loc[pos].index, ll.loc[pos, name], label=name)
    ax.set_title(pos)
    ax.set_xlabel("gameweek")
axes[0].set_ylabel("log loss (lower is better)")
axes[0].legend()
plt.tight_layout()
plt.show()

### 3. Calibration of each player's pick count

Under the model, player $j$'s pick count in round $t$ is $\text{BetaBinomial}(n_t, \alpha_{j,t}, A_t - \alpha_{j,t})$
for each posterior draw, so its posterior predictive distribution is the mixture over the draws (50 of them here).
The randomized PIT, $F(c-1) + U\,[F(c) - F(c-1)]$ with $U \sim U(0,1)$, is uniform when the model is calibrated: 50% of
the PITs in $[0.25, 0.75]$ and 90% in $[0.05, 0.95]$. A hump in the middle means the predictive distributions are too
wide, a U shape that they are too narrow, a slope that they are biased. The players who matter for the field are the
popular ones (expected share of at least 1%).

In [ ]:
pit = selection.predictive_pit(data, models)
display(pit.groupby(["position", "players"])["pit"]
        .agg(rows="size", in_50=lambda p: p.between(0.25, 0.75).mean(), in_90=lambda p: p.between(0.05, 0.95).mean(),
             mean_pit="mean").round(3))
fig, axes = plt.subplots(1, 4, figsize=(16, 3), sharey=True)
for ax, pos in zip(axes, selection.POSITIONS):
    ax.hist(pit.loc[pit["position"] == pos, "pit"], bins=10, range=(0, 1), density=True)
    ax.axhline(1, color="k", lw=1, ls="--")
    ax.set_title(f"{pos}: PIT")
plt.tight_layout()
plt.show()

### 4. Posterior predictive checks of the whole round

For every position and gameweek, 200 replicated rounds from the posterior predictive, as the likelihood generates them:
a posterior draw, **one** $p_t \sim \text{Dirichlet}(\alpha_t)$ for the round, and $n_t$ picks from it. `p_value` is the
share of replicates at least as large as the actual round; calibrated, the p-values are uniform over the rounds (about
5% below 0.05 and 5% above 0.95).

* `top5_share`, `max_share`, `hhi` (sum of the squared shares): how concentrated the picks are
* `players_picked`: how many players were picked at least once

Like the likelihood, the replicates treat the picks as independent, while a manager picks distinct players.

In [ ]:
ppc = selection.posterior_predictive(data, models)
(ppc.groupby(["stat", "position"])
 .agg(observed=("observed", "median"), replicated=("replicated", "median"), mean_p=("p_value", "mean"),
      p_below_05=("p_value", lambda p: (p < 0.05).mean()), p_above_95=("p_value", lambda p: (p > 0.95).mean()))
 .round(3))

### What this means for the team generator

The likelihood explains the concentration of a round with **one** $p_t$ shared by every manager. A team generator that
draws its own $p \sim \text{Dirichlet}(\alpha)$ for every simulated team averages that away: over a few hundred teams the
field's pick shares converge to the mean shares $\alpha/\sum\alpha$, far flatter than a real round. $\kappa$ cannot fix
this: it is fitted on the log loss of the mean shares, which are already about right ($\kappa \approx 1$). `mvt.py`
therefore draws one $p$ per position for the whole simulated field (`Settings.shared_p`); `mvt.ipynb` compares the two
generators' fields with the real ones.

In [ ]:
mean_top5 = est.groupby(["position", "round"])["pred_share"].apply(lambda s: s.nlargest(5).sum())
top5 = ppc[ppc["stat"] == "top5_share"].groupby("position")
pd.DataFrame({"actual": top5["observed"].median(), "one p per round (likelihood)": top5["replicated"].median(),
              "mean shares (one p per team)": mean_top5.groupby(level="position").median()}).loc[selection.POSITIONS].round(3)